#### This script does the following:
- formats and exports  all non tdm inputs tables as csvs
- performs a basic check (coming soon...)
- creates the policy override ("zoning_parcels_p")
- loads the input tables into the h5

In [ ]:
import arcpy
from arcpy import env
from datetime import date
import os
from arcgis import GIS
from arcgis.features import GeoAccessor
import pandas as pd
import numpy as np
from zipfile import ZipFile
import logging

arcpy.env.overwriteOutput = True
arcpy.env.parallelProcessingFactor = "90%"

# show all columns
pd.options.display.max_columns = None

# pd.DataFrame.spatial.from_featureclass(???)
# df.spatial.to_featureclass(location=???,sanitize_columns=False)

In [ ]:
d = date.today().strftime("%Y%m%d")

In [ ]:
mode = 'v900' 
# mode = 'v832' 
scenario = 'rtp'
# scenario = 'no-center'
# scenario = 'metro-urban'

In [ ]:
out_folder = f'Outputs\{mode}_Inputs_{d}'

if not os.path.exists(out_folder):
    os.makedirs(out_folder)

outputs = [out_folder, "scratch.gdb"]    

gdb = os.path.join(outputs[0], outputs[1])

if not arcpy.Exists(gdb):
    arcpy.CreateFileGDB_management(outputs[0], outputs[1])


# Step 1) Export tables from the Geodatabase

# *parcels*


In [ ]:
# get the grid ID for utility restiction
target_features = r"..\Current_Inputs\remm_base_year.gdb\parcels"
join_features = r"..\Ancillary\utility_restriction_grid.shp"
output_features = os.path.join(gdb, "_01_parcels_grid_sj")

fieldmappings = arcpy.FieldMappings()
fieldmappings.addTable(target_features)
fieldmappings.addTable(join_features)

# # max_dua
# fieldindex = fieldmappings.findFieldMapIndex('max_dua')
# fieldmap = fieldmappings.getFieldMap(fieldindex)
# fieldmap.mergeRule = 'Mean'
# fieldmappings.replaceFieldMap(fieldindex, fieldmap)

# run the spatial join
sj3 = arcpy.SpatialJoin_analysis(target_features, join_features, output_features,'JOIN_ONE_TO_ONE', "KEEP_COMMON", 
                           fieldmappings, "HAVE_THEIR_CENTER_IN")

In [ ]:
p = pd.DataFrame.spatial.from_featureclass(sj3[0])
del p['OBJECTID']
del p['TARGET_FID']
del p['Join_Count']
p['parcel_id'] = p['parcel_id'].astype(int)
print(p.shape)

(712236, 46)


# *buildings*

In [ ]:
b = pd.DataFrame.spatial.from_featureclass(r"..\Current_Inputs\remm_base_year.gdb\buildings")
del b['OBJECTID']
del b['SHAPE']

expanded_du = pd.read_csv(r"..\Current_Inputs\expanded_DU_for_BuildingTable.csv")

# expand the residential units
b_new = b.merge(expanded_du, on='parcel_id', how='left')
b_new['parcel_id'] = b_new['parcel_id'].astype(int)

# subset to buildings with a building type
b_new = b_new[(b_new['building_type_id'] >= 1)].copy()
b_new['building_id'] = b_new['building_id'].astype(int)

b_new.loc[b_new['addDU']> 0, 'expanded'] = 1
b_new.loc[b_new['addDU']> 0, 'residential_units'] = b_new['residential_units'] + b_new['addDU']


#b_new['pipeline_id'] = np.nan

# *zoning baseline*


In [ ]:
zb = pd.DataFrame.spatial.from_featureclass(r"..\Current_Inputs\remm_base_year.gdb\zoning_baseline")
del zb['SHAPE']
del zb['OBJECTID']

zb['parcel_id'] = zb['parcel_id'].astype(int)


zb_mag = pd.read_csv(r"..\Current_Inputs\New MAG Policy and Pipeline\zoning_baseline_MAG20230330.csv")
zb_mag_ids = zb_mag['parcel_id'].to_list()
zb_no_mag = zb[zb['parcel_id'].isin(zb_mag_ids) == False].copy()
zb_new = pd.concat([zb_no_mag, zb_mag])

# remap back to the old true/false format
if mode == 'v832':
    codes = {1:'t', 0:'f'}
    zb_new['type1'] = zb_new['type1'].map(codes).astype(str)
    zb_new['type2'] = zb_new['type2'].map(codes).astype(str)
    zb_new['type3'] = zb_new['type3'].map(codes).astype(str)
    zb_new['type4'] = zb_new['type4'].map(codes).astype(str)
    zb_new['type5'] = zb_new['type5'].map(codes).astype(str)
    zb_new['type6'] = zb_new['type6'].map(codes).astype(str)
    zb_new['type7'] = zb_new['type7'].map(codes).astype(str)
    zb_new['type8'] = zb_new['type8'].map(codes).astype(str)

if mode == 'v900':
    pass

# *pipeline jobs (insert into h5 later)*

In [ ]:
pj = pd.read_csv(r"..\Current_Inputs\pipelineJobs_20230405.csv")

# *pipeline buildings (insert into h5 later)*

In [ ]:
# pb = pd.read_csv(r"..\Current_Inputs\pipeline_buildings_WFRC_20230202.csv")
# pb_mag = pd.read_csv(r"..\Current_Inputs\New MAG Policy and Pipeline\pipeline_buildings_MAG20221209.csv")
# pb_new = pd.concat([pb, pb_mag])

pb_new = pd.read_csv(r"..\Current_Inputs\pipeline_buildings_20230405.csv")
del pb_new['TAZID_900']

In [ ]:
# (ZONING BASELINE) Make parcels that have pipeline building or jobs, "undevelopable"
ids_pb = pb_new[pb_new['DEVTYPE']!='demolition']['parcel_id'].to_list()
ids_pj = pj['building_id'].to_list()
pipeline_ids = set(ids_pb + ids_pj)

# step 1 set zoning baseline to no build
zb_new.loc[zb_new['parcel_id'].isin(pipeline_ids)==True, 'max_dua'] = np.nan
zb_new.loc[zb_new['parcel_id'].isin(pipeline_ids)==True, 'max_far'] = np.nan

if mode == 'v832':
    zb_new.loc[zb_new['parcel_id'].isin(pipeline_ids)==True, 'type1'] = 'f'
    zb_new.loc[zb_new['parcel_id'].isin(pipeline_ids)==True, 'type2'] = 'f'
    zb_new.loc[zb_new['parcel_id'].isin(pipeline_ids)==True, 'type3'] = 'f'
    zb_new.loc[zb_new['parcel_id'].isin(pipeline_ids)==True, 'type4'] = 'f'
    zb_new.loc[zb_new['parcel_id'].isin(pipeline_ids)==True, 'type5'] = 'f'
    zb_new.loc[zb_new['parcel_id'].isin(pipeline_ids)==True, 'type6'] = 'f'
    zb_new.loc[zb_new['parcel_id'].isin(pipeline_ids)==True, 'type7'] = 'f'
    zb_new.loc[zb_new['parcel_id'].isin(pipeline_ids)==True, 'type8'] = 'f'

if mode == 'v900':
    zb_new.loc[zb_new['parcel_id'].isin(pipeline_ids)==True, 'type1'] = 0
    zb_new.loc[zb_new['parcel_id'].isin(pipeline_ids)==True, 'type2'] = 0
    zb_new.loc[zb_new['parcel_id'].isin(pipeline_ids)==True, 'type3'] = 0
    zb_new.loc[zb_new['parcel_id'].isin(pipeline_ids)==True, 'type4'] = 0
    zb_new.loc[zb_new['parcel_id'].isin(pipeline_ids)==True, 'type5'] = 0
    zb_new.loc[zb_new['parcel_id'].isin(pipeline_ids)==True, 'type6'] = 0
    zb_new.loc[zb_new['parcel_id'].isin(pipeline_ids)==True, 'type7'] = 0
    zb_new.loc[zb_new['parcel_id'].isin(pipeline_ids)==True, 'type8'] = 0

# step 2 set parcels to no build
p.loc[p['parcel_id'].isin(pipeline_ids)==True, 'NoBuild'] = 1

In [ ]:
# (BUILDINGS) Increase job spaces in SL and Davis County where pipeline action doesn't occur
b_new = b_new.merge(p[['parcel_id', 'county_id']], on='parcel_id', how='left')
b_new.loc[(b_new['county_id'] == 11) & (b_new['job_spaces'] > 0) & (b_new['building_id'].isin(pipeline_ids)==False), 'job_spaces'] = round(b_new['job_spaces'] * 1.16)
b_new.loc[(b_new['county_id'] == 35) & (b_new['job_spaces'] > 0) & (b_new['building_id'].isin(pipeline_ids)==False), 'job_spaces'] = round(b_new['job_spaces'] * 1.085)
del b_new['county_id']

In [ ]:
# check to ensure demolition and construction dont occur in the same year on the same parcel
# remember to use redevelop code

# *zoning parcels p (AKA policy override)*

In [ ]:
# select scenario
pop = os.path.join(r"..\Current_Inputs\policy_override.gdb\policy_override_polygons")
pop_lyr = arcpy.MakeFeatureLayer_management(pop, 'pop_lyr')



# optional scenarios
if scenario == 'metro-urban':
    query = "(AreaType IN ('Metropolitan Center', 'Urban Center', 'Employment District', 'Special District', 'Industrial District') AND \
            AreaName NOT IN ('Porter Rockwell', 'Fore Lakes Center', 'Old Mill')) OR \
            AreaName IN ('Herriman Annexation', 'Herriman Porter Rockwell Center', 'Jones Ranch')"
    arcpy.SelectLayerByAttribute_management(pop_lyr, 'NEW_SELECTION', query)

elif scenario == 'no-center':
    query = "(AreaType IN ('Other', 'Employment District', 'Special District', 'Industrial District')) AND \
    (AreaName NOT IN ('Porter Rockwell', 'Fore Lakes Center', 'Old Mill')) OR \
    (AreaName IN ('Herriman Annexation', 'Herriman Porter Rockwell Center', 'Jones Ranch'))"
    arcpy.SelectLayerByAttribute_management(pop_lyr, 'NEW_SELECTION', query)

In [ ]:
# join parcels and buildings table, then export
p_sdf = p[['parcel_id','Tax_Exempt','NoBuild', 'TAZID_900','SHAPE']].copy()
b_sdf = b[['parcel_id','building_type_id2']].copy()
p_sdf.merge(b_sdf, on='parcel_id', how='left').spatial.to_featureclass(location=os.path.join(gdb, 'parcels_buildings'),sanitize_columns=False)

'e:\\Projects\\REMM-Manage-Base-Year-Data\\Generate_Policy_Override_and_Base_Year_H5\\Outputs\\v900_Inputs_20240409\\scratch.gdb\\parcels_buildings'

In [ ]:
# clean-up
del p['SHAPE']
del p_sdf
del b_sdf

In [ ]:
# create parcel pts
parcels_pts = arcpy.FeatureToPoint_management(os.path.join(gdb, 'parcels_buildings'), os.path.join(gdb, '_02_parcels_pts'), "INSIDE")
parcels_pts_lyr = arcpy.MakeFeatureLayer_management(parcels_pts, 'parcels_pts_lyr')

# excluded some special case parcels
arcpy.SelectLayerByAttribute_management(parcels_pts_lyr, 'NEW_SELECTION', "(building_type_id2 in (6,9,10,13,14,16) or Tax_Exempt = 1 or NoBuild = 1) AND (parcel_id NOT IN (13670,17117,13898,13907))")
arcpy.DeleteFeatures_management(parcels_pts_lyr)

<Result 'parcels_pts_lyr'>

In [ ]:
# use spatial join to summarize max dua
target_features = parcels_pts_lyr
join_features = pop_lyr
output_features = os.path.join(gdb, "_00_parcels_pts_policy_sj")

fieldmappings = arcpy.FieldMappings()
fieldmappings.addTable(target_features)
fieldmappings.addTable(join_features)

# max_dua
fieldindex = fieldmappings.findFieldMapIndex('max_dua')
fieldmap = fieldmappings.getFieldMap(fieldindex)
fieldmap.mergeRule = 'Mean'
fieldmappings.replaceFieldMap(fieldindex, fieldmap)

# run the spatial join
sj = arcpy.SpatialJoin_analysis(target_features, join_features, output_features,'JOIN_ONE_TO_MANY', "KEEP_COMMON", 
                           fieldmappings, "INTERSECT")

In [ ]:
parcel_pts_policy_sdf = pd.DataFrame.spatial.from_featureclass(sj[0])

# if the year attribute is missing, use phase year from RTP
parcel_pts_policy_sdf.loc[(parcel_pts_policy_sdf['phase_begin_year'] > 0) & (parcel_pts_policy_sdf['year'].isna()== True), 'year'] = parcel_pts_policy_sdf['phase_begin_year']
parcel_pts_policy_sdf = parcel_pts_policy_sdf[(parcel_pts_policy_sdf['year'] > 0)]
parcel_pts_policy_sdf = parcel_pts_policy_sdf[['parcel_id', 'TAZID_900', 'max_dua','max_far','year','type1','type2','type3','type4','type5','type6','type7','type8','locnote','mponote', 'AreaName']].copy()

In [ ]:
# Spot adjustments to big parcels within policy
parcel_pts_policy_sdf.loc[parcel_pts_policy_sdf['parcel_id'] == 35338, 'max_far'] = .25
parcel_pts_policy_sdf.loc[parcel_pts_policy_sdf['parcel_id'] == 57372, 'max_far'] = .25
parcel_pts_policy_sdf.loc[parcel_pts_policy_sdf['parcel_id'] == 48385, 'max_far'] = .2

In [ ]:
# export table
# d = date.today().strftime("%Y%m%d")
# zpp_out = os.path.join(outputs[0],f'zoning_parcels_p_{d}.csv')
# parcel_pts_policy_sdf.to_csv(zpp_out, index=False)
# print(f'file exported to:{zpp_out}')

In [ ]:
# BONUS: check for duplicate entrys
duplicate = parcel_pts_policy_sdf[parcel_pts_policy_sdf.duplicated(subset=['parcel_id', 'year'], keep =False)]
n_duplicates = duplicate.shape[0]

print(f'There were {n_duplicates} duplicates rows in the output.')

if n_duplicates > 0:
    duplicate.to_csv(os.path.join(outputs[0],f'duplicates_{d}.csv'))

There were 0 duplicates rows in the output.


In [ ]:
# join to MAG's version
zpp = parcel_pts_policy_sdf
zpp_mag = pd.read_csv(r"..\Current_Inputs\New MAG Policy and Pipeline\zoning_parcels_p_MAG20221209.csv")
zpp_new = pd.concat([zpp, zpp_mag])

# *agriculture*

In [ ]:
a = pd.DataFrame.spatial.from_featureclass(r"..\Current_Inputs\remm_base_year.gdb\agriculture")
del a['SHAPE']
a['parcel_id'] = a['parcel_id'].astype(int)

# *redev friction*

In [ ]:
rf = pd.DataFrame.spatial.from_featureclass(r"..\Current_Inputs\remm_base_year.gdb\redev_friction")
del rf['SHAPE']
del rf['OBJECTID']
rf_mag = pd.read_csv(r"..\Current_Inputs\New MAG Policy and Pipeline\redev_friction_MAG20221209.csv")
rf_mag_ids = rf_mag['parcel_id'].to_list()
print(len(rf))
print(len(rf_mag))

712239
1807


In [ ]:
rf = rf[rf['redev_friction']>0].copy()
rf_no_mag = rf[rf['parcel_id'].isin(rf_mag_ids) == False].copy()
rf_new = pd.concat([rf_no_mag, rf_mag])

rf_new['parcel_id'] = rf_new['parcel_id'].astype(int)
rf_new.shape

(5873, 2)

# *export*

In [ ]:
p.to_csv(os.path.join(outputs[0], f'parcels_{d}.csv'), index=False)
b_new.to_csv(os.path.join(outputs[0], f'buildings_{d}.csv'), index=False)
zb_new.to_csv(os.path.join(outputs[0], f'zoning_baseline_{d}.csv'), index=False)
pb_new.to_csv(os.path.join(outputs[0], f'pipeline_buildings_{d}.csv'), index=False)
zpp_new.to_csv(os.path.join(outputs[0], f'zoning_parcels_p_{d}.csv'), index=False)
rf_new.to_csv(os.path.join(outputs[0], f'redev_friction_{d}.csv'), index=False)
a.to_csv(os.path.join(outputs[0], f'agriculture_{d}.csv'), index=False)

# Step 2) Create H5 database

## *Create empty hdf5*

In [ ]:
# store path for new hdf5
d = date.today().strftime("%Y%m%d")
hdf_name = f'remm_data_2019_base_year_{d}'
new_hdf = os.path.join(outputs[0], hdf_name + '.h5')
print(new_hdf)

# if the h5 exists already delete it; it will not overwrite
if os.path.exists(new_hdf):
    try:
        new_hdf.close()
    except:
        pass
    
    os.remove(new_hdf)

# Create empty h5   
hdf = pd.HDFStore(new_hdf)   

Outputs\v900_Inputs_20240409\remm_data_2019_base_year_20240409.h5


## *Add parcels*

In [ ]:
# parcels
parcels = p

if mode == 'v832':
       parcels['zone_id'] = parcels['TAZID_832']
if mode == 'v900':
       parcels['zone_id'] = parcels['TAZID_900']

       # set district ids
       taz9 = pd.DataFrame.spatial.from_featureclass('..\Ancillary\TAZ.gdb\TAZ_900')[['SA_TAZID', 'DISTLRG', 'DISTMED', 'DISTSML']]
       parcels = parcels.merge(taz9, left_on='TAZID_900', right_on='SA_TAZID', how='left')
       parcels['distsml_id'] = parcels['DISTSML']
       parcels['distmed_id'] = parcels['DISTMED']
       parcels['distlrg_id'] = parcels['DISTLRG']
       parcels.drop(list(taz9.columns), axis=1, inplace=True)
       del taz9

parcels['parcel_sqft'] = parcels['parcel_sqft'].astype(float)

parcels.rename({'parcel_sqft':'shape_area'}, axis=1, inplace=True) # delete later

# parcels['parcel_id_REMM'] = parcels['parcel_id']
# parcels.rename({'COUNTY_ID':'county_id'}, axis=1, inplace=True)

# convert back to square feet for now, since the model is coded to used those units (parcel_acres is derived from shape_area)
# parcels['shape_area'] = parcels['shape_area'] * 10.7639
# parcels['shape_area'] = parcels['shape_area'].round(0).astype(int)

if mode == 'v832':
       parcels = parcels[['zone_id', 'CO_NAME', 'county_id', 'land_value',
              'x', 'y', 'Split', 'parcel_acres',
              'TAZID_832', 'TAZID_900', 'Old_PID', 'parent_parcel', 'elevation',
              'fwy_exit', 'airport', 'rail_depot', 'stream', 'trail', 'university',
              'shape_area', 'volume_one_way', 'volume_two_way', 'airport_distance',
              'fwy_exit_dist', 'raildepot_dist', 'university_dist', 'trail_dist',
              'stream_dist', 'train_station', 'rail_stn_dist', 'bus_rte_dist',
              'bus_stop', 'bus_stop_dist', 'volume_two_way_nofwy', 'distsml_id',
              'distmed_id', 'distlrg_id', 'zonal_ppa', 'parcel_id','grid_id']].copy()

# if mode == 'v900':
#        parcels = parcels[['zone_id', 'CO_NAME', 'county_id', 'land_value',
#               'x', 'y', 'Split', 'parcel_acres',
#               'TAZID_832', 'TAZID_900', 'Old_PID', 'parent_parcel', 'elevation',
#               'fwy_exit', 'airport', 'rail_depot', 'stream', 'trail', 'university',
#               'parcel_sqft', 'volume_one_way', 'volume_two_way', 'airport_distance',
#               'fwy_exit_dist', 'raildepot_dist', 'university_dist', 'trail_dist',
#               'stream_dist', 'train_station', 'rail_stn_dist', 'bus_rte_dist',
#               'bus_stop', 'bus_stop_dist', 'volume_two_way_nofwy', 'distsml_id',
#               'distmed_id', 'distlrg_id', 'zonal_ppa', 'parcel_id']].copy()
       
if mode == 'v900':
       parcels = parcels[['zone_id', 'CO_NAME', 'county_id', 'land_value',
              'x', 'y', 'Split', 'parcel_acres',
              'TAZID_832', 'TAZID_900', 'Old_PID', 'parent_parcel', 'elevation',
              'fwy_exit', 'airport', 'rail_depot', 'stream', 'trail', 'university',
              'shape_area', 'volume_one_way', 'volume_two_way', 'airport_distance',
              'fwy_exit_dist', 'raildepot_dist', 'university_dist', 'trail_dist',
              'stream_dist', 'train_station', 'rail_stn_dist', 'bus_rte_dist',
              'bus_stop', 'bus_stop_dist', 'volume_two_way_nofwy', 'distsml_id',
              'distmed_id', 'distlrg_id', 'zonal_ppa', 'parcel_id','grid_id']].copy()

parcels.columns

Index(['zone_id', 'CO_NAME', 'county_id', 'land_value', 'x', 'y', 'Split',
       'parcel_acres', 'TAZID_832', 'TAZID_900', 'Old_PID', 'parent_parcel',
       'elevation', 'fwy_exit', 'airport', 'rail_depot', 'stream', 'trail',
       'university', 'shape_area', 'volume_one_way', 'volume_two_way',
       'airport_distance', 'fwy_exit_dist', 'raildepot_dist',
       'university_dist', 'trail_dist', 'stream_dist', 'train_station',
       'rail_stn_dist', 'bus_rte_dist', 'bus_stop', 'bus_stop_dist',
       'volume_two_way_nofwy', 'distsml_id', 'distmed_id', 'distlrg_id',
       'zonal_ppa', 'parcel_id', 'grid_id'],
      dtype='object')

In [ ]:
parcels[parcels.index.duplicated()]

,zone_id,CO_NAME,county_id,land_value,x,y,Split,parcel_acres,TAZID_832,TAZID_900,Old_PID,parent_parcel,elevation,fwy_exit,airport,rail_depot,stream,trail,university,shape_area,volume_one_way,volume_two_way,airport_distance,fwy_exit_dist,raildepot_dist,university_dist,trail_dist,stream_dist,train_station,rail_stn_dist,bus_rte_dist,bus_stop,bus_stop_dist,volume_two_way_nofwy,distsml_id,distmed_id,distlrg_id,zonal_ppa,parcel_id,grid_id


In [ ]:
hdf.put('parcels', parcels.set_index('parcel_id'), format='t', data_columns=True)

## *Add buildings*

In [ ]:
buildings = b_new
buildings_lu = {0:None,
                1:1,
                2:2,
                3:3,
                4:4,
                5:5,
                6:6,
                7:7,
                8:8,
                9:6, 
                10:8,
                11:8,
                12:None,
                13:5,
                14:None,
                15:None,
                16:None,
                99:None}

# remap building types
buildings['building_type_id'] = buildings['building_type_id'].map(buildings_lu)

######################################
# fill in some fake square footage and years (comment out later)
######################################

# fill in some fake numbers
buildings.loc[(buildings['year_built'].isna() ==True) | (buildings['year_built'] == 0), 'year_built'] =  np.random.randint(1900, 2019, buildings[(buildings['year_built'].isna() ==True) | (buildings['year_built'] == 0)].shape[0])


# add  floor to area ratio * parcel acres instead!!!!
buildings.loc[((buildings['building_sqft'].isna() ==True) | (buildings['building_sqft'] == 0)) & (buildings['building_type_id'] == 3), 
              'building_sqft'] =  np.random.randint(10000, 200000, buildings[((buildings['building_sqft'].isna() ==True) | (buildings['building_sqft'] == 0)) & (buildings['building_type_id'] == 3)].shape[0])


# add  floor to area ratio * parcel acres instead!!!!
buildings.loc[((buildings['building_sqft'].isna() ==True) | (buildings['building_sqft'] == 0)) & (buildings['building_type_id'] != 3), 
              'building_sqft'] =  np.random.randint(1400, 14000, buildings[((buildings['building_sqft'].isna() ==True) | (buildings['building_sqft'] == 0)) & (buildings['building_type_id'] != 3)].shape[0])

buildings.loc[((buildings['non_residential_sqft'].isna() ==True) | (buildings['non_residential_sqft'] == 0)) & (buildings['building_type_id'].isin([1,2]) == False), 
              'non_residential_sqft'] = buildings['building_sqft']

buildings.loc[((buildings['non_residential_sqft'].isna() ==True) | (buildings['non_residential_sqft'] == 0)) & (buildings['building_type_id'].isin([1,2]) == True), 
              'non_residential_sqft'] = 0

buildings.loc[(buildings['residential_units'].isna() ==True), 
              'residential_units'] =  0

buildings.loc[(buildings['unit_price_non_residential'].isna() ==True), 
              'unit_price_non_residential'] =  0
buildings['unit_price_non_residential'] = buildings['unit_price_non_residential'].astype(float)

mean = buildings['res_price_per_sqft'].mean()
buildings.loc[((buildings['res_price_per_sqft'].isna() ==True) | (buildings['res_price_per_sqft'] == 0)) & (buildings['building_type_id'].isin([1,2]) == True), 
              'res_price_per_sqft'] = mean

buildings.replace([np.inf, -np.inf], 0, inplace=True)

buildings.loc[((buildings['res_price_per_sqft'].isna() ==True) | (buildings['res_price_per_sqft'] == 0)) & (buildings['building_type_id'].isin([1,2]) == False), 
              'res_price_per_sqft'] = 0
####################################################

# # subset to buildings with a building type
# buildings = buildings[buildings['building_type_id'] >= 1].copy()
# buildings['building_id'] = buildings['building_id'].astype(int)

# delete extra columns
del buildings['building_type_id2']
del buildings['basebldg']
del buildings['building_type']
del buildings['expanded']
del buildings['addDU']

buildings.columns

Index(['parcel_id', 'building_id', 'building_sqft', 'building_type_id',
       'non_residential_sqft', 'note', 'residential_units', 'stories',
       'unit_price_non_residential', 'year_built', 'res_price_per_sqft',
       'job_spaces'],
      dtype='object')

In [ ]:
hdf.put('buildings', buildings.set_index('building_id'), format='t', data_columns=True)

## *Add jobs*

In [ ]:
jobs = pd.read_csv(r"..\Current_Inputs\jobs_20230501.csv")
jobs_for_h5 = jobs.copy()

if mode == 'v900':
    del jobs_for_h5['cid']

jobs_for_h5.columns

Index(['jobs_id', 'building_id', 'sector_id'], dtype='object')

In [ ]:
hdf.put('jobs', jobs_for_h5.set_index('jobs_id'), format='t', data_columns=True)

## *Add households*

In [ ]:
households = pd.read_csv(r"..\Current_Inputs\households_20230501.csv")
hh_for_h5 = households.copy()

if mode == 'v900':
    del hh_for_h5['cid']
    
hh_for_h5.columns

Index(['household_id', 'cars', 'household_type_id', 'persons', 'income',
       'workers', 'children', 'age_of_head', 'race_id', 'familyhh', 'block_id',
       'building_id'],
      dtype='object')

In [ ]:
hdf.put('households', hh_for_h5, format='t', data_columns=True)

## *Add travel data*

In [ ]:
if mode == 'v832':
    travel_data = pd.read_csv(r"..\Current_Inputs\travel_data_2015.csv")
if mode == 'v900':
    travel_data = pd.read_csv(r"..\Current_Inputs\travel_data_2019.csv")
    del travel_data['Unnamed: 0']
travel_data.columns

Index(['from_zone_id', 'to_zone_id', 'travel_time', 'travel_time_transit',
       'log0', 'log1', 'log2'],
      dtype='object')

In [ ]:
hdf.put('travel_data', travel_data.set_index(['from_zone_id', 'to_zone_id']), format='t', data_columns=True)

## *Add zoning baseline*

In [ ]:
zoning_baseline = zb_new
zoning_baseline.columns

Index(['parcel_id', 'max_dua', 'max_far', 'max_height', 'type1', 'type2',
       'type3', 'type4', 'type5', 'type6', 'type7', 'type8'],
      dtype='object')

In [ ]:
zoning_baseline.loc[zoning_baseline['max_dua']==0, 'max_dua'] = np.nan
zoning_baseline.loc[zoning_baseline['max_far']==0, 'max_far'] = np.nan
hdf.put('zoning_baseline', zoning_baseline.set_index('parcel_id'), format='t', data_columns=True)

In [ ]:
zoning_baseline['type1'].value_counts()

1.0    548875
0.0    163361
Name: type1, dtype: int64

In [ ]:
# close open files
hdf.close()

# Step 3) Summary and Checks

In [ ]:
b_real = buildings.merge(p, on='parcel_id', how='left')
parcels_buildings = p.merge(b, on='parcel_id', how='left')
jobs_by_cid_and_sector = jobs.groupby(['cid','sector_id'])[['jobs_id']].count().reset_index()
hh_with_cid = households.merge(p, left_on='building_id', right_on='parcel_id', how='left')

In [ ]:
# logging
logfile = os.path.join(outputs[0], f'REMM_Base_Year_Data_Summary_{d}.txt')
f = open(logfile, 'w')
f.write(f'REMM Base Year Data Summary {d}\n\n')

38

In [ ]:
# parcels
f.write('--PARCELS--\n\n')
f.write('All Counties:\n')
f.write(f"\tNumber of Parcels: {p.shape[0]}\n")
f.write(f"\tVacant Buildable Land: {round(parcels_buildings[parcels_buildings['building_type_id2'] == 0]['parcel_acres'].sum(), 2)}\n")
f.write('\n')
f.write('Weber County:\n')
f.write(f"\tNumber of Parcels: {p[p['county_id'] == 57].shape[0]}\n")
f.write(f"\tVacant Buildable Land: {round(parcels_buildings[(parcels_buildings['county_id'] == 57) & (parcels_buildings['building_type_id2'] == 0) ]['parcel_acres'].sum(), 2)}\n")
f.write('\n')
f.write('Davis County:\n')
f.write(f"\tNumber of Parcels: {p[p['county_id'] == 11].shape[0]}\n")
f.write(f"\tVacant Buildable Land: {round(parcels_buildings[(parcels_buildings['county_id'] == 11) & (parcels_buildings['building_type_id2'] == 0) ]['parcel_acres'].sum(), 2)}\n")
f.write('\n')
f.write('Salt Lake County:\n')
f.write(f"\tNumber of Parcels: {p[p['county_id'] == 35].shape[0]}\n")
f.write(f"\tVacant Buildable Land: {round(parcels_buildings[(parcels_buildings['county_id'] == 35) & (parcels_buildings['building_type_id2'] == 0) ]['parcel_acres'].sum(), 2)}\n")
f.write('\n')
f.write('Utah County:\n')
f.write(f"\tNumber of Parcels: {p[p['county_id'] == 49].shape[0]}\n")
f.write(f"\tVacant Buildable Land: {round(parcels_buildings[(parcels_buildings['county_id'] == 49) & (parcels_buildings['building_type_id2'] == 0) ]['parcel_acres'].sum(), 2)}\n")
f.write('\n')

# buildings
f.write('--BUILDINGS--\n\n')
f.write('All Counties:\n')
f.write(f"\tNumber of Buildings: {b_real.shape[0]}\n")
f.write(f"\tResidential Units: {round(b_real['residential_units'].sum())}\n")
f.write(f"\tJob Spaces: {round(b_real['job_spaces'].sum())}\n")
f.write('\n')
f.write('Weber County:\n')
f.write(f"\tNumber of Buildings: {b_real[b_real['county_id'] == 57].shape[0]}\n")
f.write(f"\tResidential Units: {round(b_real[b_real['county_id'] == 57]['residential_units'].sum())}\n")
f.write(f"\tJob Spaces: {round(b_real[b_real['county_id'] == 57]['job_spaces'].sum())}\n")
f.write('\n')
f.write('Davis County:\n')
f.write(f"\tNumber of Buildings: {b_real[b_real['county_id'] == 11].shape[0]}\n")
f.write(f"\tResidential Units: {round(b_real[b_real['county_id'] == 11]['residential_units'].sum())}\n")
f.write(f"\tJob Spaces: {round(b_real[b_real['county_id'] == 11]['job_spaces'].sum())}\n")
f.write('\n')
f.write('Salt Lake County:\n')
f.write(f"\tNumber of Buildings: {b_real[b_real['county_id'] == 35].shape[0]}\n")
f.write(f"\tResidential Units: {round(b_real[b_real['county_id'] == 35]['residential_units'].sum())}\n")
f.write(f"\tJob Spaces: {round(b_real[b_real['county_id'] == 35]['job_spaces'].sum())}\n")
f.write('\n')
f.write('Utah County:\n')
f.write(f"\tNumber of Buildings: {b_real[b_real['county_id'] == 49].shape[0]}\n")
f.write(f"\tResidential Units: {round(b_real[b_real['county_id'] == 49]['residential_units'].sum())}\n")
f.write(f"\tJob Spaces: {round(b_real[b_real['county_id'] == 49]['job_spaces'].sum())}\n")
f.write('\n')

# jobs
f.write('--JOBS--\n\n')
f.write('All Counties:\n')
f.write(f"\tNumber of Jobs: {jobs.shape[0]}\n")
# f.write(f"\tNumber of Jobs (Sector 1): {jobs_by_cid_and_sector[(jobs_by_cid_and_sector['sector_id'] == 1)].values[0][2]}\n")
# f.write(f"\tNumber of Jobs (Sector 3): {jobs_by_cid_and_sector[(jobs_by_cid_and_sector['sector_id'] == 3)].values[0][2]}\n")
# f.write(f"\tNumber of Jobs (Sector 4): {jobs_by_cid_and_sector[(jobs_by_cid_and_sector['sector_id'] == 4)].values[0][2]}\n")
# f.write(f"\tNumber of Jobs (Sector 5): {jobs_by_cid_and_sector[(jobs_by_cid_and_sector['sector_id'] == 5)].values[0][2]}\n")
# f.write(f"\tNumber of Jobs (Sector 6): {jobs_by_cid_and_sector[(jobs_by_cid_and_sector['sector_id'] == 6)].values[0][2]}\n")
# f.write(f"\tNumber of Jobs (Sector 7): {jobs_by_cid_and_sector[(jobs_by_cid_and_sector['sector_id'] == 7)].values[0][2]}\n")
# f.write(f"\tNumber of Jobs (Sector 9): {jobs_by_cid_and_sector[(jobs_by_cid_and_sector['sector_id'] == 9)].values[0][2]}\n")
# f.write(f"\tNumber of Jobs (Sector 10): {jobs_by_cid_and_sector[(jobs_by_cid_and_sector['sector_id'] == 10)].values[0][2]}\n")

f.write('\n')
f.write('Weber County:\n')
f.write(f"\tNumber of Jobs (Sector 1): {jobs_by_cid_and_sector[(jobs_by_cid_and_sector['cid'] == 57) & (jobs_by_cid_and_sector['sector_id'] == 1)].values[0][2]}\n")
f.write(f"\tNumber of Jobs (Sector 3): {jobs_by_cid_and_sector[(jobs_by_cid_and_sector['cid'] == 57) & (jobs_by_cid_and_sector['sector_id'] == 3)].values[0][2]}\n")
f.write(f"\tNumber of Jobs (Sector 4): {jobs_by_cid_and_sector[(jobs_by_cid_and_sector['cid'] == 57) & (jobs_by_cid_and_sector['sector_id'] == 4)].values[0][2]}\n")
f.write(f"\tNumber of Jobs (Sector 5): {jobs_by_cid_and_sector[(jobs_by_cid_and_sector['cid'] == 57) & (jobs_by_cid_and_sector['sector_id'] == 5)].values[0][2]}\n")
f.write(f"\tNumber of Jobs (Sector 6): {jobs_by_cid_and_sector[(jobs_by_cid_and_sector['cid'] == 57) & (jobs_by_cid_and_sector['sector_id'] == 6)].values[0][2]}\n")
f.write(f"\tNumber of Jobs (Sector 7): {jobs_by_cid_and_sector[(jobs_by_cid_and_sector['cid'] == 57) & (jobs_by_cid_and_sector['sector_id'] == 7)].values[0][2]}\n")
f.write(f"\tNumber of Jobs (Sector 9): {jobs_by_cid_and_sector[(jobs_by_cid_and_sector['cid'] == 57) & (jobs_by_cid_and_sector['sector_id'] == 9)].values[0][2]}\n")
f.write(f"\tNumber of Jobs (Sector 10): {jobs_by_cid_and_sector[(jobs_by_cid_and_sector['cid'] == 57) & (jobs_by_cid_and_sector['sector_id'] == 10)].values[0][2]}\n")
f.write(f"\tNumber of Jobs (Total): {jobs[jobs['cid'] == 57].shape[0]}\n")
f.write('\n')
f.write('Davis County:\n')
f.write(f"\tNumber of Jobs (Sector 1): {jobs_by_cid_and_sector[(jobs_by_cid_and_sector['cid'] == 11) & (jobs_by_cid_and_sector['sector_id'] == 1)].values[0][2]}\n")
f.write(f"\tNumber of Jobs (Sector 3): {jobs_by_cid_and_sector[(jobs_by_cid_and_sector['cid'] == 11) & (jobs_by_cid_and_sector['sector_id'] == 3)].values[0][2]}\n")
f.write(f"\tNumber of Jobs (Sector 4): {jobs_by_cid_and_sector[(jobs_by_cid_and_sector['cid'] == 11) & (jobs_by_cid_and_sector['sector_id'] == 4)].values[0][2]}\n")
f.write(f"\tNumber of Jobs (Sector 5): {jobs_by_cid_and_sector[(jobs_by_cid_and_sector['cid'] == 11) & (jobs_by_cid_and_sector['sector_id'] == 5)].values[0][2]}\n")
f.write(f"\tNumber of Jobs (Sector 6): {jobs_by_cid_and_sector[(jobs_by_cid_and_sector['cid'] == 11) & (jobs_by_cid_and_sector['sector_id'] == 6)].values[0][2]}\n")
f.write(f"\tNumber of Jobs (Sector 7): {jobs_by_cid_and_sector[(jobs_by_cid_and_sector['cid'] == 11) & (jobs_by_cid_and_sector['sector_id'] == 7)].values[0][2]}\n")
f.write(f"\tNumber of Jobs (Sector 9): {jobs_by_cid_and_sector[(jobs_by_cid_and_sector['cid'] == 11) & (jobs_by_cid_and_sector['sector_id'] == 9)].values[0][2]}\n")
f.write(f"\tNumber of Jobs (Sector 10): {jobs_by_cid_and_sector[(jobs_by_cid_and_sector['cid'] == 11) & (jobs_by_cid_and_sector['sector_id'] == 10)].values[0][2]}\n")
f.write(f"\tNumber of Jobs (Total): {jobs[jobs['cid'] == 11].shape[0]}\n")
f.write('\n')
f.write('Salt Lake County:\n')
f.write(f"\tNumber of Jobs (Sector 1): {jobs_by_cid_and_sector[(jobs_by_cid_and_sector['cid'] == 35) & (jobs_by_cid_and_sector['sector_id'] == 1)].values[0][2]}\n")
f.write(f"\tNumber of Jobs (Sector 3): {jobs_by_cid_and_sector[(jobs_by_cid_and_sector['cid'] == 35) & (jobs_by_cid_and_sector['sector_id'] == 3)].values[0][2]}\n")
f.write(f"\tNumber of Jobs (Sector 4): {jobs_by_cid_and_sector[(jobs_by_cid_and_sector['cid'] == 35) & (jobs_by_cid_and_sector['sector_id'] == 4)].values[0][2]}\n")
f.write(f"\tNumber of Jobs (Sector 5): {jobs_by_cid_and_sector[(jobs_by_cid_and_sector['cid'] == 35) & (jobs_by_cid_and_sector['sector_id'] == 5)].values[0][2]}\n")
f.write(f"\tNumber of Jobs (Sector 6): {jobs_by_cid_and_sector[(jobs_by_cid_and_sector['cid'] == 35) & (jobs_by_cid_and_sector['sector_id'] == 6)].values[0][2]}\n")
f.write(f"\tNumber of Jobs (Sector 7): {jobs_by_cid_and_sector[(jobs_by_cid_and_sector['cid'] == 35) & (jobs_by_cid_and_sector['sector_id'] == 7)].values[0][2]}\n")
f.write(f"\tNumber of Jobs (Sector 9): {jobs_by_cid_and_sector[(jobs_by_cid_and_sector['cid'] == 35) & (jobs_by_cid_and_sector['sector_id'] == 9)].values[0][2]}\n")
f.write(f"\tNumber of Jobs (Sector 10): {jobs_by_cid_and_sector[(jobs_by_cid_and_sector['cid'] == 35) & (jobs_by_cid_and_sector['sector_id'] == 10)].values[0][2]}\n")
f.write(f"\tNumber of Jobs (Total): {jobs[jobs['cid'] == 35].shape[0]}\n")
f.write('\n')
f.write('Utah County:\n')
f.write(f"\tNumber of Jobs (Sector 1): {jobs_by_cid_and_sector[(jobs_by_cid_and_sector['cid'] == 49) & (jobs_by_cid_and_sector['sector_id'] == 1)].values[0][2]}\n")
f.write(f"\tNumber of Jobs (Sector 3): {jobs_by_cid_and_sector[(jobs_by_cid_and_sector['cid'] == 49) & (jobs_by_cid_and_sector['sector_id'] == 3)].values[0][2]}\n")
f.write(f"\tNumber of Jobs (Sector 4): {jobs_by_cid_and_sector[(jobs_by_cid_and_sector['cid'] == 49) & (jobs_by_cid_and_sector['sector_id'] == 4)].values[0][2]}\n")
f.write(f"\tNumber of Jobs (Sector 5): {jobs_by_cid_and_sector[(jobs_by_cid_and_sector['cid'] == 49) & (jobs_by_cid_and_sector['sector_id'] == 5)].values[0][2]}\n")
f.write(f"\tNumber of Jobs (Sector 6): {jobs_by_cid_and_sector[(jobs_by_cid_and_sector['cid'] == 49) & (jobs_by_cid_and_sector['sector_id'] == 6)].values[0][2]}\n")
f.write(f"\tNumber of Jobs (Sector 7): {jobs_by_cid_and_sector[(jobs_by_cid_and_sector['cid'] == 49) & (jobs_by_cid_and_sector['sector_id'] == 7)].values[0][2]}\n")
f.write(f"\tNumber of Jobs (Sector 9): {jobs_by_cid_and_sector[(jobs_by_cid_and_sector['cid'] == 49) & (jobs_by_cid_and_sector['sector_id'] == 9)].values[0][2]}\n")
f.write(f"\tNumber of Jobs (Sector 10): {jobs_by_cid_and_sector[(jobs_by_cid_and_sector['cid'] == 49) & (jobs_by_cid_and_sector['sector_id'] == 10)].values[0][2]}\n")
f.write(f"\tNumber of Jobs (Total): {jobs[jobs['cid'] == 49].shape[0]}\n")
f.write('\n')

# households
f.write('--HOUSEHOLDS--\n\n')
f.write('All Counties:\n')
f.write(f"\tNumber of Households: {hh_with_cid.shape[0]}\n")
f.write('\n')
f.write('Weber County:\n')
f.write(f"\tNumber of Households: {hh_with_cid[hh_with_cid['county_id'] == 57].shape[0]}\n")
f.write('\n')
f.write('Davis County:\n')
f.write(f"\tNumber of Households: {hh_with_cid[hh_with_cid['county_id'] == 11].shape[0]}\n")
f.write('\n')
f.write('Salt Lake County:\n')
f.write(f"\tNumber of Households: {hh_with_cid[hh_with_cid['county_id'] == 35].shape[0]}\n")
f.write('\n')
f.write('Utah County:\n')
f.write(f"\tNumber of Households: {hh_with_cid[hh_with_cid['county_id'] == 49].shape[0]}\n")
f.write('\n\n')


2

In [ ]:
ids_p_from_pb = pb_new['parcel_id'].to_list()
ids_p_from_p = p['parcel_id'].to_list()
ids_p_from_b = b['parcel_id'].to_list()
ids_b_from_b = b_new['building_id'].to_list()
ids_b_from_hh = households['building_id'].to_list()
ids_b_from_j = jobs['building_id'].to_list()
ids_zb = zb_new['parcel_id'].to_list()
ids_rf = rf_new['parcel_id'].to_list()
ids_a = a['parcel_id'].to_list()

In [ ]:
# ensure no index ids for any table are duplicated
f.write('--CHECK FOR DUPLICATES--\n\n')

if len(ids_p_from_p) == len(set(ids_p_from_p)):
    f.write('No duplicate parcel ids in parcels\n')
    print('No duplicates ids in parcels')

if len(ids_b_from_b) == len(set(ids_b_from_b)):
    f.write('No duplicate building ids in buildings\n')
    print('No duplicates ids in buildings')

if len(ids_zb) == len(set(ids_zb)):
    f.write('No duplicates parcel ids in zoning baseline\n')
    print('No duplicates ids in zoning baseline')

if len(ids_rf) == len(set(ids_rf)):
    f.write('No duplicate parcel ids in redev friction\n\n')
    print('No duplicates ids in redev friction')

No duplicates ids in parcels
No duplicates ids in buildings
No duplicates ids in zoning baseline
No duplicates ids in redev friction


In [ ]:
# make sure there are not external taz ids in parcel file (coming soon...)

In [ ]:
# ensure all child tables id is present in their parent table

# ensure there is zb for every p
p_ids_in_zb_missing_from_parcels = list(set(ids_zb) - set(ids_p_from_p))
if len(p_ids_in_zb_missing_from_parcels) > 0:
    print('p_ids_in_zb_missing_from_parcels')
    print(p_ids_in_zb_missing_from_parcels)

p_ids_in_parcels_missing_from_zb = list(set(ids_p_from_p) - set(ids_zb))
if len(p_ids_in_parcels_missing_from_zb) > 0:
    print('p_ids_in_parcels_missing_from_zb')
    print(p_ids_in_parcels_missing_from_zb)

# ensure all b pids are in p
b_ids_in_buildings_missing_from_parcels = list(set(ids_p_from_b) - set(ids_p_from_p))
if len(b_ids_in_buildings_missing_from_parcels) > 0:
    print('b_ids_in_buildings_missing_from_parcels')
    print(b_ids_in_buildings_missing_from_parcels)

p_ids_in_parcels_missing_from_buildings = list(set(ids_p_from_p) - set(ids_p_from_b))
if len(p_ids_in_parcels_missing_from_buildings) > 0:
    print('p_ids_in_parcels_missing_from_buildings')
    print(p_ids_in_parcels_missing_from_buildings)

# ensure all rf pids are in p
p_ids_in_rf_missing_from_parcels = list(set(ids_rf) - set(ids_p_from_p))
if len(p_ids_in_rf_missing_from_parcels) > 0:
    print('p_ids_in_rf_missing_from_parcels')
    print(p_ids_in_rf_missing_from_parcels)

# ensure all agriculture pids are in p
p_ids_in_a_missing_from_parcels = list(set(ids_a) - set(ids_p_from_p))
if len(p_ids_in_a_missing_from_parcels) > 0:
    print('p_ids_in_a_missing_from_parcels')
    print(p_ids_in_a_missing_from_parcels)

# ensure all jobs bids are in b
b_ids_in_jobs_missing_from_buildings = list(set(ids_b_from_j) - set(ids_b_from_b))
if len(b_ids_in_jobs_missing_from_buildings) > 0:
    print('b_ids_in_jobs_missing_from_buildings')
    print(b_ids_in_jobs_missing_from_buildings)

# ensure all households bids are in b
b_ids_in_hh_missing_from_buildings = list(set(ids_b_from_hh) - set(ids_b_from_b))
if len(b_ids_in_hh_missing_from_buildings) > 0:
    print('b_ids_in_hh_missing_from_buildings')
    print(b_ids_in_hh_missing_from_buildings)

# ensure all pipeline buildings pids are in p
p_ids_in_pb_missing_from_parcels = list(set(ids_p_from_pb) - set(ids_p_from_p))
if len(p_ids_in_pb_missing_from_parcels) > 0:
    print('p_ids_in_pb_missing_from_parcels')
    print(p_ids_in_pb_missing_from_parcels)

# Summarize missing data in parcels and buildings
# Make sure max dua values make sense with building types allowed!!!!
# Example type 1 only allowed, max dua should be between .5 and 1
# Make sure max fars make sense with building types allowed!!!!
# ensure jobs count per building does not exceed job spaces
# add county controls and comparison
# ensure county id/cid from parcel/buildings, hh and jobs table all match

In [ ]:
f.close()